In [1]:
import os
import json
from pathlib import Path
from dotenv import load_dotenv
from datetime import datetime
from typing import Any, Dict, List
from sqlalchemy import Integer, JSON, String, Text, func, text
from sqlalchemy.orm import DeclarativeBase, declared_attr, Mapped, mapped_column
from sqlalchemy.ext.asyncio import AsyncAttrs, async_sessionmaker, create_async_engine
from pgvector.sqlalchemy import Vector
from langchain_openai import ChatOpenAI
from langchain_gigachat.chat_models import GigaChat
from langchain_gigachat import GigaChatEmbeddings
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.messages import SystemMessage, HumanMessage

In [12]:
NOTEBOOK_DIR = Path.cwd()
dotenv_path = NOTEBOOK_DIR / '.env'
load_dotenv(dotenv_path=str(dotenv_path))

OPENROUTER_API_KEY = os.environ.get("OPENROUTER_API_KEY")
GIGACHAT_API_KEY = os.environ.get("GIGACHAT_API_KEY")

POSTGRES_DB_NAME = "userdb"

POSTGRES_URL = (
    f"postgresql+asyncpg://user"
    f":user"
    f"@localhost"
    f":5436"
    f"/{POSTGRES_DB_NAME}"
)

## Тест

In [13]:
import requests
import secrets
import base64
import uuid
import urllib3

In [ ]:
llm = GigaChat(
    credentials=GIGACHAT_API_KEY,
    verify_ssl_certs=False,
    model="GigaChat:latest",
    scope="GIGACHAT_API_B2B",
    temperature=0.1
)

messages = [
    HumanMessage(content="Привет"),
]

res = await llm.ainvoke(messages)
res

AIMessage(content='Привет. Как настроение?', additional_kwargs={}, response_metadata={'token_usage': {'prompt_tokens': 12, 'completion_tokens': 7, 'total_tokens': 19, 'precached_prompt_tokens': 2}, 'model_name': 'GigaChat:2.0.28.2', 'x_headers': {'x-request-id': 'd48530f5-ac91-4cfc-a34e-635ebe198b64', 'x-session-id': '88f9a80b-9fd4-4e9d-a871-63e723bc41c9', 'x-client-id': None}, 'finish_reason': 'stop'}, id='d48530f5-ac91-4cfc-a34e-635ebe198b64', usage_metadata={'output_tokens': 7, 'input_tokens': 12, 'total_tokens': 19, 'input_token_details': {'cache_read': 2}})

In [15]:
embedder = GigaChatEmbeddings(credentials=GIGACHAT_API_KEY, scope="GIGACHAT_API_B2B", verify_ssl_certs=False)

embeddings = embedder.embed_query("Привет мир!")

embeddings

ResponseError: (URL('https://gigachat.devices.sberbank.ru/api/v1/embeddings'), 402, b'{"status":402,"message":"Payment Required"}\n', Headers({'server': 'SynGX', 'date': 'Thu, 25 Dec 2025 13:56:56 GMT', 'content-type': 'application/json; charset=utf-8', 'content-length': '44', 'connection': 'keep-alive', 'access-control-allow-credentials': 'true', 'access-control-allow-headers': 'Origin, X-Requested-With, Content-Type, Accept, Authorization', 'access-control-allow-methods': 'GET, POST, DELETE, OPTIONS', 'access-control-allow-origin': 'https://beta.saluteai.sberdevices.ru', 'x-request-id': '6d196e5a-1f76-4271-bc52-4c0ca084985a', 'x-session-id': '5e0f02b2-35b8-4230-b317-0274d59166ca', 'allow': 'GET, POST', 'strict-transport-security': 'max-age=31536000; includeSubDomains'}))

## Основной

In [3]:
PROMPT = '''
Вы — эксперт по парсингу XML-документов в структурированный формат JSON.
Ваша задача: извлечь данные из XML и преобразовать их в заданную структуру.

### Инструкции:
1. Проанализируйте XML-документ и определите его структуру.
2. Извлеките все сущности (events, courses, etc.) с их атрибутами.
3. Разбейте текстовое содержимое каждой сущности на логические чанки (макс. 500 символов).
4. Сохраните все связи между сущностями (например, организатор мероприятия).
5. Убедитесь, что все обязательные поля заполнены.

### Целевая структура (JSON):
{
  "metadata": {
    "source": "название_файла.xml",
    "parsing_timestamp": "ISO-формат",
    "version": "1.0"
  },
  "entities": [
    {
      "id": "уникальный_идентификатор",
      "type": "event|course|news|material",
      "attributes": {
        "title": "строка",
        "description": "строка",
        "date": "ISO-дата",
        ...
      },
      "relationships": [
        {
          "target_id": "идентификатор_связанной_сущности",
          "type": "organizer|location|..."
        }
      ],
      "content": {
        "raw": "полный_текст",
        "chunks": [
          {"text": "чанк_текста_1"},
          {"text": "чанк_текста_2"}
        ]
      }
    }
  ]
}

### Примеры:
- Для `<event>`: type="event", атрибуты включают дату, место, организатора.
- Для `<course>`: type="course", атрибуты включают преподавателя, кредиты, расписание.

### Валидация:
- Все идентификаторы должны быть уникальны.
- Все обязательные поля (помеченные *) должны быть заполнены.
- Чанки не должны превышать 500 символов.

### Формат ответа:
Верните только валидный JSON без комментариев.
'''

In [4]:
SYSTEM_PROMPT_DR = '''
Ты — высокоточный ассистент-аналитик по нормативной документации. Твоя задача — отвечать на вопросы пользователя ТОЛЬКО на основе предоставленных фрагментов документов в формате XML.

### СТРУКТУРА ДАННЫХ
Ты работаешь со специальным форматом XML, оптимизированным для поиска:
1. <document source="...">: Корневой элемент, указывает имя файла или документа.
2. <section title="...">: Логический раздел документа.
3. <text context="..."> и <list_item context="...">: Содержат основной текст.
4. <table>: Табличные данные, где <header> — это заголовки, а <cell> — данные.

### КРИТИЧЕСКИ ВАЖНОЕ ПРАВИЛО: АТРИБУТ 'CONTEXT'
Текст внутри тегов может быть коротким или оторванным от смысла (например, "Срок: 3 дня").
Истинный смысл содержится в атрибуте 'context'.
- Атрибут 'context' — это полный иерархический путь (хлебные крошки) от названия документа до конкретного абзаца.
- Ты ОБЯЗАН читать 'context' перед тем, как интерпретировать текст внутри тега.

Пример:
Вход: <text context="Приказ №10 > Приложение 1 (Стипендии) > Сроки выплат">Не позднее 25 числа</text>
Твоя интерпретация: "Согласно Приложению 1 к Приказу №10, стипендии выплачиваются не позднее 25 числа".

### ПРАВИЛА ОБРАБОТКИ ТАБЛИЦ
1. Данные в тегах <cell> внутри <row> соответствуют заголовкам <header> по их порядковому номеру.
2. Если в таблице пустая ячейка, ищи значение в контексте раздела.

### ИНСТРУКЦИЯ ПО ГЕНЕРАЦИИ ОТВЕТА
1. Сначала проанализируй вопрос пользователя.
2. Найди релевантные фрагменты в предоставленном XML, опираясь на атрибут 'context' и содержимое текста.
3. Сформируй ответ, ссылаясь на конкретные документы (источник берется из начала строки 'context' или атрибута 'source').
4. Если во фрагментах есть противоречия, отдавай приоритет документу, чей 'context' выглядит более специфичным для данного вопроса (например, "Правила для аспирантов" важнее "Общих правил").
5. Если информации в XML недостаточно для ответа, прямо скажи: "В предоставленных документах нет информации по этому вопросу". НЕ ДОДУМЫВАЙ ФАКТЫ.
'''

USER_PROMPT_DR = '''
Вопрос пользователя:
"""
{user_question}
"""

Ниже приведены фрагменты документов из Базы Знаний, которые могут содержать ответ. Данные представлены в формате XML.

<knowledge_base>
{retrieved_xml_chunks}
</knowledge_base>

Рассуждай шаг за шагом:
1. Определи, какие из фрагментов XML относятся к теме вопроса, внимательно читая атрибут 'context'.
2. Игнорируй фрагменты, где 'context' не соответствует сути вопроса, даже если ключевые слова совпадают.
3. Сформулируй полный и точный ответ. Обязательно указывай названия документов или разделов, откуда взята информация.

Твой ответ:
'''

In [5]:
llm = GigaChat(
    credentials=GIGACHAT_API_KEY,
    verify_ssl_certs=False,
    model="GigaChat:latest",
    temperature=0.1
)

In [6]:
data_dir_path = r"..\..\resources\raw\test_xml_data"
file_path = os.path.join(data_dir_path, "01_Glavnyi_korpus_Dzerzhinskogo17.xml")

with open(file_path, "r", encoding="utf-8") as f:
    xml_content = f.read()

# messages = [
#     SystemMessage(content=PROMPT),
#     HumanMessage(content=xml_content),
# ]

prompt = ChatPromptTemplate.from_messages([
    ("system", SYSTEM_PROMPT_DR),
    ("user", USER_PROMPT_DR)
])

In [7]:
# chain = prompt | llm

# response = await chain.ainvoke({
#     "user_question": "Краткая характеристика действующего порядка предоставления на объекте услуг населению",
#     "retrieved_xml_chunks": xml_content
# })

# response.content

## Эмбеддинги

In [8]:
class Base(AsyncAttrs, DeclarativeBase):
    __abstract__ = True
    __table_args__ = {'schema': POSTGRES_DB_NAME}

    id: Mapped[int] = mapped_column(Integer, primary_key=True, autoincrement=True)
    created_at: Mapped[datetime] = mapped_column(server_default=func.now())
    updated_at: Mapped[datetime] = mapped_column(server_default=func.now(), onupdate=func.now())

    @declared_attr.directive
    def __tablename__(cls) -> str:
        return cls.__name__.lower() + 's'


In [9]:
class Document(Base):
    message_number: Mapped[int] = mapped_column(Integer, nullable=False)
    content: Mapped[str] = mapped_column(Text, nullable=False)
    embedding: Mapped[List[float]] = mapped_column(Vector(1024), nullable=False)


In [10]:
engine = None
async_session_maker = None


def connection(method):
    async def wrapper(*args, **kwargs):
        async with async_session_maker() as session:
            try:
                async with session.begin():
                    return await method(*args, session=session, **kwargs)
            except Exception as e:
                await session.rollback()
                raise
            finally:
                await session.close()

    return wrapper


async def init_db():
    global engine, async_session_maker

    engine = create_async_engine(url=POSTGRES_URL)
    async_session_maker = async_sessionmaker(engine, expire_on_commit=False)


async def create_tables():
    if not engine:
        raise Exception("[create_tables] База данных PostgreSQL не инициализирована")

    async with engine.begin() as conn:
        await conn.execute(text("CREATE EXTENSION IF NOT EXISTS vector;"))
        await conn.execute(text(f"CREATE SCHEMA IF NOT EXISTS {POSTGRES_DB_NAME}"))
        await conn.run_sync(Base.metadata.create_all)


In [11]:
await init_db()

In [12]:
await create_tables()

In [13]:
import glob
xml_files = glob.glob(os.path.join(data_dir_path, "*.xml"))

In [14]:
PROMPT_CONVERTER = '''
Вы — эксперт по преобразованию XML-документов в чистый читаемый текст. Ваша задача: извлечь всю значимую информацию из XML и представить её в естественной форме.

## ИНСТРУКЦИИ:
1. **Игнорируйте XML-теги**: Удалите все `<tag>`, `</tag>`, `<?xml`, атрибуты и форматирование
2. **Сохраните структуру**: Используйте заголовки, списки, абзацы для логической группировки
3. **Извлеките все данные**: Названия, авторы, даты, цены, категории — всё важное
4. **Сделайте читаемым**: Натуральный текст как в книге или отчёте
5. **НЕ добавляйте интерпретации**: Только данные из XML

## ФОРМАТ ВЫВОДА:
- **Заголовки** для основных разделов (H1, H2)
- **Списки** для коллекций (книги, события)
- **Таблицы** для структурированных данных
- **Чистый текст** без markdown-блоков или кода

## ПРИМЕРЫ:
XML: `<book category="classic"><title>Война и мир</title><author>Лев Толстой</author></book>`
→ **Классическая литература**
   - Война и мир, Лев Толстой

XML: `<library><book><price>500</price></book></library>`
→ **Библиотека**
   Цена книги: 500 руб.

## XML ДЛЯ ОБРАБОТКИ:
{input_xml}

## РЕЗУЛЬТАТ:
Преобразуйте в чистый текст ниже:
'''

In [15]:
prompt = ChatPromptTemplate.from_messages([
    ("system", PROMPT_CONVERTER)
])

In [16]:
embedder = GigaChatEmbeddings(
    credentials=GIGACHAT_API_KEY,
    verify_ssl_certs=False
)

In [17]:
@connection
async def add_docs(session=None):
    docs = []
    
    for idx, file in enumerate(xml_files):
        with open(file, "r", encoding="utf-8") as f:
            xml_content = f.read()

        chain = prompt | llm

        response = await chain.ainvoke({"input_xml": xml_content})
        content = response.content[:500]
        embeddings = embedder.embed_query(content)
        
        doc = Document(
            message_number=idx,
            content=content,
            embedding=embeddings,
        )
        docs.append(doc)

    session.add_all(docs)
    await session.commit()

In [18]:
await add_docs()

ReadTimeout: 

In [ ]:
from sqlalchemy import select

@connection
async def vector_search_orm(embedding: List[float], top_k: int = 5, session = None):
    stmt = select(Document).order_by(
        Document.embedding.l2_distance(embedding)  # SQL: embedding <-> [0.1,0.1,...]
    ).limit(top_k)

    result = await session.execute(stmt)
    return result.scalars().all()


In [ ]:
user_request = "бюджет"
user_request_emb = embedder.embed_query(user_request)
results = await vector_search_orm(user_request_emb, top_k=2)

for doc in results:
    print(doc.content)

### Федеральное государственное бюджетное образовательное учреждение высшего профессионального образования  
**«Костромской государственный технологический университет»**

#### Основные мероприятия деятельности университета на 2015–2016 учебный год  
**Твердил ректор А. Титунин, 2015 г.**

---

#### Организационные мероприятия  
1. Реализация программных мероприятий по повышению эффективности деятельности вуза, разработка детализированного плана мероприятий оптимизации деятельности вуза на 2016 
# Основные характеристики объекта и предоставляемых услуг

## Адрес и описание объекта
Адрес объекта:  
156000, Костромская область, Костромской район, город Кострома, площадь Советская, дом 2а (объект культурного наследия: «Здание городской думы»).

Наименование предоставляемой услуги: образовательная деятельность.
Форма оказания услуг: дистанционная.
Категория обслуживаемого населения: взрослые трудоспособного возраста.

## Доступность объекта для инвалидов
### Недостатки в обеспечении доступ